# Обучение ML-моделей в Google Colab

## Подготовка (сделать один раз локально):
```bash
# 1. Экспортировать свечи из локальной БД в CSV
python -m scripts.export_candles_csv

# 2. Загрузить папку ml/data/ на Google Drive
#    Структура: MyDrive/we_trust_data/SBER.csv, GAZP.csv, ..., USDRUB.csv
```

## После обучения:
```bash
# Скопировать веса с сервера (после скачивания из Drive)
scp ml/weights/ensemble_*_v2.pkl botuser@185.197.75.125:~/we_trust_in_people_saas/ml/weights/
scp ml/weights/best_params_*_v2.json botuser@185.197.75.125:~/we_trust_in_people_saas/ml/weights/
scp ml/weights/feature_cols_*_v2.json botuser@185.197.75.125:~/we_trust_in_people_saas/ml/weights/
```

## 1. Установка зависимостей

In [ ]:
# Установка TA-Lib (требует компиляции C-библиотеки)
!apt-get install -y -q ta-lib
!pip install -q ta-lib

# Основные зависимости
!pip install -q \
    lightgbm \
    optuna \
    scikit-learn \
    pandas \
    numpy \
    structlog \
    pydantic \
    pydantic-settings \
    python-dotenv

print('Зависимости установлены')

## 2. Клонирование репозитория

In [ ]:
import os

REPO_DIR = '/content/we_trust_in_people'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/LehaDeev/we_trust_in_people.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'Рабочая папка: {os.getcwd()}')

## 3. Создание .env для Colab (без API-ключей)

In [ ]:
env_content = """
# Заглушки (не используются при обучении из CSV)
TINKOFF_TOKEN=dummy
TINKOFF_ACCOUNT_ID=0
TINKOFF_SANDBOX=true
TELEGRAM_BOT_TOKEN=dummy
POSTGRES_HOST=localhost
POSTGRES_PORT=5432
POSTGRES_DB=dummy
POSTGRES_USER=postgres
POSTGRES_PASSWORD=dummy
REDIS_HOST=localhost
REDIS_PORT=6379
REDIS_PASSWORD=

# Настройки приложения
LOG_LEVEL=INFO
DEBUG=false

# Сбор данных
DATA_TICKERS=SBER,GAZP,LKOH,YDEX,NVTK,GMKN,MGNT,TATN,ROSN,MTSS
DATA_CANDLE_INTERVAL=1h
DATA_HISTORY_DAYS=1460
DATA_START_DATE=2022-03-09
DATA_USDRUB_FIGI=BBG0013HGFT4
DATA_COLLECT_PAUSE_SECONDS=30

# ML
ML_MODEL_VERSION=v2
ML_LOOKAHEAD=2
ML_THRESHOLD=0.005
ML_N_SPLITS=5
ML_RANDOM_STATE=42
ML_OPTUNA_TRIALS_LGBM=50
ML_OPTUNA_TRIALS_XGB=50
ML_OPTUNA_TRIALS_ET=50
ML_OPTUNA_TRIALS_SVC=20
ML_OPTUNA_TRIALS_CATBOOST=30
ML_MIN_CANDLES_PREDICT=250
ML_FORCE_TUNE=false
ML_FEATURE_IMPORTANCE_THRESHOLD=0.01
ML_PRINT_FEATURE_IMPORTANCE=false

# Торговля (не используется при обучении)
TRADING_ENABLED=false
TRADING_CONFIDENCE_THRESHOLD=0.58
TRADING_LOTS_PER_TICKER=1
TRADING_STOP_LOSS_PCT=0.01
TRADING_TAKE_PROFIT_PCT=0.015
TRADING_MAX_POSITIONS=5
TRADING_INTERVAL_SECONDS=1800
TRADING_BROKER_COMMISSION_PCT=0.003
TRADING_TAX_PCT=0.13
TRADING_DIVIDEND_PROTECTION_DAYS=1
TRADING_DIVIDEND_OVERRIDE=
TRADING_VOLUME_MIN_RATIO=1.3
RETRAIN_ENABLED=false
RETRAIN_HOUR=2
RETRAIN_MINUTE=0
RETRAIN_TIMEZONE=Europe/Moscow
RETRAIN_FORCE_TUNE=false
"""

with open('.env', 'w') as f:
    f.write(env_content.strip())

print('.env создан')

## 4. Подключение Google Drive и загрузка CSV

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Папка на Drive, куда вы загрузили CSV-файлы
# Измените путь если нужно
DRIVE_DATA_DIR = '/content/drive/MyDrive/we_trust_data'

import os
csv_files = [f for f in os.listdir(DRIVE_DATA_DIR) if f.endswith('.csv')]
print(f'Найдено CSV файлов: {len(csv_files)}')
for f in sorted(csv_files):
    size_mb = os.path.getsize(f'{DRIVE_DATA_DIR}/{f}') / 1024 / 1024
    print(f'  {f}: {size_mb:.1f} MB')

## 5. Запуск обучения

In [ ]:
import asyncio
import sys
sys.path.insert(0, REPO_DIR)

from pathlib import Path
from config.settings import ml_settings
from ml.train import train_model

data_dir = Path(DRIVE_DATA_DIR)

# force_tune=True — полный перебор гиперпараметров (занимает 2-4 часа)
# force_tune=False — использует кеш best_params_*.json если он есть в weights/
results = await train_model(force_tune=True, data_dir=data_dir)

print(f'\nОбучено тикеров: {len(results)}')
for ticker, path in results.items():
    print(f'  {ticker}: {path.name}')

## 6. Просмотр результатов

In [ ]:
import json
from pathlib import Path

results_path = Path('ml/weights/last_results.json')
if results_path.exists():
    data = json.loads(results_path.read_text())
    print(f"Обучено: {data['trained_at']}")
    print(f"force_tune: {data['force_tune']}")
    print()
    print(f"{'Тикер':<10} {'F1':>8}")
    print('-' * 20)
    for ticker, f1 in sorted(data['f1_scores'].items(), key=lambda x: -x[1]):
        bar = '#' * int(f1 * 40)
        print(f"{ticker:<10} {f1:>8.4f}  {bar}")
    if data['failed']:
        print(f"\nОшибки: {data['failed']}")

## 7. Сохранение весов на Google Drive

In [ ]:
import shutil
from pathlib import Path

DRIVE_WEIGHTS_DIR = '/content/drive/MyDrive/saas/ml/weights'
Path(DRIVE_WEIGHTS_DIR).mkdir(parents=True, exist_ok=True)

weights_dir = Path('ml/weights')
copied = 0
for pattern in ('ensemble_*_v2.pkl', 'best_params_*_v2.json', 'features_*_v2.json', 'last_results.json'):
    for src in weights_dir.glob(pattern):
        dst = Path(DRIVE_WEIGHTS_DIR) / src.name
        shutil.copy2(src, dst)
        copied += 1
        print(f'  Скопирован: {src.name}')

print(f'\nВсего скопировано: {copied} файлов → {DRIVE_WEIGHTS_DIR}')
print('\nСкачайте папку с Drive и скопируйте на сервер:')
print('  scp ml/weights/*.pkl botuser@185.197.75.125:~/we_trust_in_people_saas/ml/weights/')
print('  scp ml/weights/*.json botuser@185.197.75.125:~/we_trust_in_people_saas/ml/weights/')